# ==========================================================
# Breast Cancer Survival Prediction using Apache Spark
# Notebook 02: Data Preprocessing
# ==========================================================

"""
Objective
---------
1. Clean the raw SEER breast cancer dataset.
2. Handle SEER special codes.
3. Handle missing values.
4. Remove invalid records.
5. Standardize variables.
6. Prepare a clean dataset for feature engineering.

Note
----
No feature engineering.
No model training.
No model evaluation.
"""

In [47]:
# 1. Import Libraries | Khai báo thư viện

print("=" * 60)
print("1. IMPORT LIBRARIES")
print("=" * 60)

import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Add project root directory to path | Thêm thư mục gốc dự án vào hệ thống
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Import custom functions from src | Nạp các hàm tự định nghĩa từ src
from src.data.loader import create_spark_session, load_csv

1. IMPORT LIBRARIES


In [48]:
# 2. Create Spark Session | Khởi tạo phiên làm việc Spark

print("=" * 60)
print("2. CREATE SPARK SESSION")
print("=" * 60)

spark = create_spark_session("SEER Breast Cancer Data Preprocessing")

2. CREATE SPARK SESSION


In [49]:
# 3. Load Raw CSV Dataset | Tải tập dữ liệu CSV thô

print("=" * 60)
print("3. LOAD RAW CSV DATASET")
print("=" * 60)

df = load_csv(spark, "../data/raw/breast_cancer_seer-2004-2015.csv")

# Store original dimensions for validation | Lưu trữ kích thước gốc để kiểm tra chéo
original_row_count = df.count()
original_column_count = len(df.columns)

print(f"Original Rows    : {original_row_count:,}")
print(f"Original Columns : {original_column_count}")

3. LOAD RAW CSV DATASET


Original Rows    : 457,351
Original Columns : 29


In [50]:
# 4. Review Dataset Structure | Khảo sát cấu trúc tập dữ liệu

print("=" * 60)
print("4. RAW DATASET OVERVIEW")
print("=" * 60)

df.printSchema()
print()
df.show(10, truncate=False)

4. RAW DATASET OVERVIEW
root
 |-- Age recode with <1 year olds and 90+: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Race recode (W, B, AI, API): string (nullable = true)
 |-- Marital status at diagnosis: string (nullable = true)
 |-- CS tumor size (2004-2015): integer (nullable = true)
 |-- Survival months: string (nullable = true)
 |-- Vital status recode (study cutoff used): string (nullable = true)
 |-- Grade Recode (thru 2017): string (nullable = true)
 |-- PR Status Recode Breast Cancer (1990+): string (nullable = true)
 |-- ER Status Recode Breast Cancer (1990+): string (nullable = true)
 |-- Breast - Adjusted AJCC 6th T (1988-2015): string (nullable = true)
 |-- Breast - Adjusted AJCC 6th N (1988-2015): string (nullable = true)
 |-- Regional nodes examined (1988+): integer (nullable = true)
 |-- Regional nodes positive (1988+): integer (nullable = true)
 |-- Sequence number: string (nullable = true)
 |-- Patient ID: integer (nullable = true)
 |-- Primary Sit

In [51]:
# 5. Rename Columns | Đổi tên các cột thuộc tính

print("=" * 60)
print("5. RENAME COLUMNS")
print("=" * 60)

column_mapping = {
    "Age recode with <1 year olds and 90+": "Age",
    "Sex": "Sex",
    "Race recode (W, B, AI, API)": "Race",
    "Marital status at diagnosis": "Marital_Status",
    "CS tumor size (2004-2015)": "Tumor_Size",
    "Survival months": "Survival_Months",
    "Vital status recode (study cutoff used)": "Vital_Status",
    "Grade Recode (thru 2017)": "Grade",
    "PR Status Recode Breast Cancer (1990+)": "PR_Status",
    "ER Status Recode Breast Cancer (1990+)": "ER_Status",
    "Breast - Adjusted AJCC 6th T (1988-2015)": "AJCC_T",
    "Breast - Adjusted AJCC 6th N (1988-2015)": "AJCC_N",
    "Breast - Adjusted AJCC 6th M (1988-2015)": "AJCC_M",
    "Breast - Adjusted AJCC 6th Stage (1988-2015)": "AJCC_Stage",
    "Regional nodes examined (1988+)": "Regional_Nodes_Examined",
    "Regional nodes positive (1988+)": "Regional_Nodes_Positive",
    "Sequence number": "Sequence_Number",
    "Patient ID": "Patient_ID",
    "Primary Site": "Primary_Site",
    "Histologic Type ICD-O-3": "Histologic_Type",
    "Behavior recode for analysis": "Behavior",
    "Laterality": "Laterality",
    "Diagnostic Confirmation": "Diagnostic_Confirmation",
    "Lymph-vascular Invasion (2004+ varying by schema)": "Lymph_Vascular_Invasion",
    "RX Summ--Surg Prim Site (1998-2022)": "Surgery_Primary_Site",
    "RX Summ--Surg Oth Reg/Dis (2003+)": "Surgery_Other_Regional",
    "RX Summ--Surg/Rad Seq": "Surgery_Radiation_Sequence",
    "Radiation recode": "Radiation",
    "Chemotherapy recode (yes, no/unk)": "Chemotherapy"
}

for old_col, new_col in column_mapping.items():
    if old_col in df.columns:
        df = df.withColumnRenamed(old_col, new_col)

print("Column renaming completed.")

5. RENAME COLUMNS
Column renaming completed.


In [52]:
# 6. Verify Column Renaming | Xác minh đổi tên cột thành công

print("=" * 60)
print("6. VERIFY COLUMN RENAMING")
print("=" * 60)
print("Current columns in DataFrame:")
print(df.columns)
print("Total number of columns:", len(df.columns))

6. VERIFY COLUMN RENAMING
Current columns in DataFrame:
['Age', 'Sex', 'Race', 'Marital_Status', 'Tumor_Size', 'Survival_Months', 'Vital_Status', 'Grade', 'PR_Status', 'ER_Status', 'AJCC_T', 'AJCC_N', 'Regional_Nodes_Examined', 'Regional_Nodes_Positive', 'Sequence_Number', 'Patient_ID', 'Primary_Site', 'Histologic_Type', 'Behavior', 'Laterality', 'Diagnostic_Confirmation', 'AJCC_M', 'Lymph_Vascular_Invasion', 'Surgery_Primary_Site', 'Surgery_Other_Regional', 'Surgery_Radiation_Sequence', 'Radiation', 'Chemotherapy', 'AJCC_Stage']
Total number of columns: 29


In [53]:
# 7. Check SEER Coding | Kiểm tra mã phân phối dữ liệu SEER

check_columns = [
    "Tumor_Size",
    "Survival_Months",
    "Regional_Nodes_Examined",
    "Regional_Nodes_Positive"
]

for c in check_columns:
    print("\n" + "=" * 50)
    print(f"Value distribution for column: {c}")
    print("=" * 50)
    df.groupBy(c).count().orderBy(F.desc("count")).show(15, False)


Value distribution for column: Tumor_Size
+----------+-----+
|Tumor_Size|count|
+----------+-----+
|15        |27858|
|999       |27129|
|25        |19416|
|10        |19217|
|20        |18944|
|12        |18893|
|30        |15857|
|8         |14055|
|18        |14035|
|11        |13546|
|9         |12793|
|13        |12424|
|7         |11585|
|14        |11248|
|17        |11191|
+----------+-----+
only showing top 15 rows


Value distribution for column: Survival_Months
+---------------+-----+
|Survival_Months|count|
+---------------+-----+
|0000           |4160 |
|0098           |3800 |
|0102           |3718 |
|0105           |3710 |
|0096           |3676 |
|0100           |3613 |
|0097           |3581 |
|0107           |3578 |
|0099           |3512 |
|0101           |3509 |
|0104           |3499 |
|0108           |3452 |
|0103           |3433 |
|0116           |3415 |
|0110           |3344 |
+---------------+-----+
only showing top 15 rows


Value distribution for column: Regional

In [54]:
# 8. Handle SEER Special Codes | Xử lý mã đặc biệt của hệ SEER

print("\n" + "=" * 60)
print("8. HANDLING SEER SPECIAL CODES TO NULL")
print("=" * 60)

# Convert numeric special codes to NULL | Chuyển đổi mã số đặc biệt về NULL
numeric_special_codes = {
    "Tumor_Size": [990, 991, 992, 993, 994, 995, 996, 997, 998, 999],
    "Regional_Nodes_Examined": [90, 95, 96, 97, 98, 99],
    "Regional_Nodes_Positive": [90, 95, 96, 97, 98, 99]
}

for column, codes in numeric_special_codes.items():
    if column in df.columns:
        count_before = df.filter(F.col(column).isin(codes)).count()
        df = df.withColumn(
            column,
            F.when(F.col(column).isin(codes), None).otherwise(F.col(column))
        )
        print(f"{column:<30}: Converted {count_before:,} numeric special codes to NULL")

# Convert string special codes to NULL | Chuyển đổi mã chuỗi đặc biệt về NULL
string_special_codes = [
    "Unknown", "Unknown reason", "Blank(s)", "NA", "Not Applicable", "Unspecified"
]

for column, dtype in df.dtypes:
    if dtype == "string":
        count_before = df.filter(F.col(column).isin(string_special_codes)).count()
        if count_before > 0:
            df = df.withColumn(
                column,
                F.when(F.col(column).isin(string_special_codes), None).otherwise(F.col(column))
            )
            print(f"{column:<30}: Converted {count_before:,} string special codes to NULL")     


8. HANDLING SEER SPECIAL CODES TO NULL
Tumor_Size                    : Converted 37,004 numeric special codes to NULL
Regional_Nodes_Examined       : Converted 19,874 numeric special codes to NULL
Regional_Nodes_Positive       : Converted 73,774 numeric special codes to NULL
Race                          : Converted 2,499 string special codes to NULL
Marital_Status                : Converted 22,144 string special codes to NULL
Survival_Months               : Converted 2,753 string special codes to NULL
Grade                         : Converted 49,354 string special codes to NULL
AJCC_T                        : Converted 485 string special codes to NULL
AJCC_N                        : Converted 485 string special codes to NULL
Diagnostic_Confirmation       : Converted 4,412 string special codes to NULL
AJCC_M                        : Converted 485 string special codes to NULL
Lymph_Vascular_Invasion       : Converted 457,351 string special codes to NULL
AJCC_Stage                    : 

In [55]:
# 9. Missing Check | Kiểm tra giá trị khuyết thiếu sau xử lý mã

print("\n" + "=" * 60)
print("9. NULL CHECK AFTER SPECIAL CODE HANDLING")
print("=" * 60)

for column in df.columns:
    count = df.filter(F.col(column).isNull()).count()
    if count > 0:
        print(f"{column:40} {count:,} nulls")


9. NULL CHECK AFTER SPECIAL CODE HANDLING
Race                                     2,499 nulls
Marital_Status                           22,144 nulls
Tumor_Size                               37,004 nulls
Survival_Months                          2,753 nulls
Grade                                    49,354 nulls
AJCC_T                                   485 nulls
AJCC_N                                   485 nulls
Regional_Nodes_Examined                  19,874 nulls
Regional_Nodes_Positive                  73,774 nulls
Diagnostic_Confirmation                  4,412 nulls
AJCC_M                                   485 nulls
Lymph_Vascular_Invasion                  457,351 nulls
AJCC_Stage                               485 nulls


In [56]:
# 10. Missing Value Percentage | Tính tỷ lệ phần trăm khuyết thiếu

print("=" * 60)
print("10. MISSING VALUE PERCENTAGE")
print("=" * 60)

total_rows = df.count()
for column in df.columns:
    missing_count = df.filter(F.col(column).isNull()).count()
    if missing_count > 0:
        percentage = (missing_count / total_rows) * 100
        print(f"{column:40} {missing_count:10,} ({percentage:6.2f}%)")

10. MISSING VALUE PERCENTAGE
Race                                          2,499 (  0.55%)
Marital_Status                               22,144 (  4.84%)
Tumor_Size                                   37,004 (  8.09%)
Survival_Months                               2,753 (  0.60%)
Grade                                        49,354 ( 10.79%)
AJCC_T                                          485 (  0.11%)
AJCC_N                                          485 (  0.11%)
Regional_Nodes_Examined                      19,874 (  4.35%)
Regional_Nodes_Positive                      73,774 ( 16.13%)
Diagnostic_Confirmation                       4,412 (  0.96%)
AJCC_M                                          485 (  0.11%)
Lymph_Vascular_Invasion                     457,351 (100.00%)
AJCC_Stage                                      485 (  0.11%)


In [57]:
# 11. Handle Missing Values | Điền khuyết giá trị phân loại bằng "Unknown"

print("=" * 60)
print("11. HANDLING MISSING VALUES")
print("=" * 60)

# Fill selected categorical NULLs with "Unknown" | Điền khuyết biến phân loại bằng "Unknown"
categorical_columns = [
    "Race",
    "Marital_Status",
    "Grade",
    "Diagnostic_Confirmation"
]

for column in categorical_columns:
    if column in df.columns:
        df = df.fillna({column: "Unknown"})
        
print("Categorical missing values successfully filled with 'Unknown'.")

11. HANDLING MISSING VALUES
Categorical missing values successfully filled with 'Unknown'.


In [58]:
# 12. Validate Missing Values | Xác thực các giá trị khuyết còn lại

print("=" * 60)
print("12. MISSING VALUE VALIDATION")
print("=" * 60)

remaining_missing = False
for column in df.columns:
    count = df.filter(F.col(column).isNull()).count()
    if count > 0:
        remaining_missing = True
        print(f"{column:40} {count:,} remaining nulls")

if not remaining_missing:
    print("Success: No missing values remain in the dataset.")

12. MISSING VALUE VALIDATION
Tumor_Size                               37,004 remaining nulls
Survival_Months                          2,753 remaining nulls
AJCC_T                                   485 remaining nulls
AJCC_N                                   485 remaining nulls
Regional_Nodes_Examined                  19,874 remaining nulls
Regional_Nodes_Positive                  73,774 remaining nulls
AJCC_M                                   485 remaining nulls
Lymph_Vascular_Invasion                  457,351 remaining nulls
AJCC_Stage                               485 remaining nulls


In [59]:
# 13. Remove Invalid Records | Loại bỏ các bản ghi không hợp lệ

print("=" * 60)
print("13. REMOVING INVALID RECORDS")
print("=" * 60)

before_count = df.count()

# Filter out physical anomalies in Tumor and Node attributes | Loại bỏ ngoại lệ vật lý của u và hạch
df = df.filter(
    (F.col("Tumor_Size").isNull()) |
    ((F.col("Tumor_Size") > 0) & (F.col("Tumor_Size") < 989))
)

df = df.filter(
    (F.col("Regional_Nodes_Examined").isNull()) | (F.col("Regional_Nodes_Examined") >= 0)
)

df = df.filter(
    (F.col("Regional_Nodes_Positive").isNull()) | (F.col("Regional_Nodes_Positive") >= 0)
)

after_count = df.count()
print(f"Number of invalid SEER records removed: {before_count - after_count:,}")

13. REMOVING INVALID RECORDS
Number of invalid SEER records removed: 1,264


In [60]:
# 14. Outlier Check | Khảo sát thông số thống kê ngoại lệ

print("=" * 60)
print("14. OUTLIER CHECK")
print("=" * 60)

numerical_columns = [
    "Tumor_Size",
    "Regional_Nodes_Examined",
    "Regional_Nodes_Positive"
]

for column in numerical_columns:
    if column in df.columns:
        print(f"\nDescriptive summary for column: {column}")
        df.select(column).summary().show()

14. OUTLIER CHECK

Descriptive summary for column: Tumor_Size
+-------+------------------+
|summary|        Tumor_Size|
+-------+------------------+
|  count|            419083|
|   mean| 24.07345084386625|
| stddev|24.369512695507026|
|    min|                 1|
|    25%|                11|
|    50%|                18|
|    75%|                30|
|    max|               988|
+-------+------------------+


Descriptive summary for column: Regional_Nodes_Examined
+-------+-----------------------+
|summary|Regional_Nodes_Examined|
+-------+-----------------------+
|  count|                 436394|
|   mean|      5.996496285466803|
| stddev|     6.9772890748489385|
|    min|                      0|
|    25%|                      1|
|    50%|                      3|
|    75%|                      9|
|    max|                     87|
+-------+-----------------------+


Descriptive summary for column: Regional_Nodes_Positive
+-------+-----------------------+
|summary|Regional_Nodes_Positive

In [61]:
# 15. Convert Data Types | Ép kiểu thuộc tính thống nhất

print("=" * 60)
print("15. CONVERTING DATA TYPES")
print("=" * 60)

# Cast selected numeric float fields to Integer | Ép kiểu các thuộc tính số về kiểu nguyên
numeric_columns = [
    "Tumor_Size",
    "Regional_Nodes_Examined",
    "Regional_Nodes_Positive"
]

for column in numeric_columns:
    if column in df.columns:
        df = df.withColumn(column, F.col(column).cast("int"))

print("Data type conversion to Integer completed.")

# Validate updated Schema | Kiểm tra lại lược đồ thuộc tính sau chuyển đổi
print("\n" + "=" * 60)
print("UPDATED SCHEMA")
print("=" * 60)
df.printSchema()

df.select("AJCC_T", "AJCC_N", "AJCC_M", "AJCC_Stage").distinct().show(20, False)

15. CONVERTING DATA TYPES
Data type conversion to Integer completed.

UPDATED SCHEMA
root
 |-- Age: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Race: string (nullable = false)
 |-- Marital_Status: string (nullable = false)
 |-- Tumor_Size: integer (nullable = true)
 |-- Survival_Months: string (nullable = true)
 |-- Vital_Status: string (nullable = true)
 |-- Grade: string (nullable = false)
 |-- PR_Status: string (nullable = true)
 |-- ER_Status: string (nullable = true)
 |-- AJCC_T: string (nullable = true)
 |-- AJCC_N: string (nullable = true)
 |-- Regional_Nodes_Examined: integer (nullable = true)
 |-- Regional_Nodes_Positive: integer (nullable = true)
 |-- Sequence_Number: string (nullable = true)
 |-- Patient_ID: integer (nullable = true)
 |-- Primary_Site: integer (nullable = true)
 |-- Histologic_Type: integer (nullable = true)
 |-- Behavior: string (nullable = true)
 |-- Laterality: string (nullable = true)
 |-- Diagnostic_Confirmation: string (nullable = 

In [62]:
# 16. Standardize Category Labels | Đồng bộ nhãn danh mục thô về 'Unknown'

print("=" * 60)
print("16. STANDARDIZING CATEGORY LABELS")
print("=" * 60)

columns_to_standardize = [
    "Race",
    "Marital_Status",
    "Grade",
    "Diagnostic_Confirmation"
]

for column in columns_to_standardize:
    if column in df.columns:
        df = df.withColumn(
            column,
            F.when(F.col(column).isin(["Blank(s)", "Unknown reason", "NA"]), "Unknown")
            .otherwise(F.col(column))
        )

print("Standardization of category labels to 'Unknown' completed.")

16. STANDARDIZING CATEGORY LABELS
Standardization of category labels to 'Unknown' completed.


In [63]:
# 17. Detect Constant Features | Phát hiện các đặc trưng không đổi để xem xét

print("=" * 60)
print("17. CONSTANT FEATURE DETECTION")
print("=" * 60)

for column in df.columns:
    unique_count = df.select(column).distinct().count()
    if unique_count == 1:
        print(f"Warning: Constant feature detected: {column}")

17. CONSTANT FEATURE DETECTION


In [64]:
# 18. Remove Unnecessary Columns & Prevent Target Leakage

print("=" * 60)
print("18. REMOVING UNNECESSARY COLUMNS & PREVENTING TARGET LEAKAGE")
print("=" * 60)

# Drop redundant fields and Survival_Months to avoid Target Leakage | Loại bỏ biến thừa và biến số tháng sống sót
remove_columns = [
    "Patient_ID",
    "Primary_Site",
    "Behavior",
    "Lymph_Vascular_Invasion",
    "Survival_Months"  # Dropped because it contains post-diagnosis info | Loại bỏ do chứa thông tin sau chẩn đoán
]

df = df.drop(*remove_columns)
print(f"Successfully dropped columns: {remove_columns}")

18. REMOVING UNNECESSARY COLUMNS & PREVENTING TARGET LEAKAGE
Successfully dropped columns: ['Patient_ID', 'Primary_Site', 'Behavior', 'Lymph_Vascular_Invasion', 'Survival_Months']


In [65]:
# 19. Validate Clean Dataset | Đánh giá tổng thể dữ liệu sạch

print("=" * 60)
print("19. CLEAN DATASET VALIDATION")
print("=" * 60)

print(f"Final Rows    : {df.count():,}")
print(f"Final Columns : {len(df.columns)}")
df.printSchema()

19. CLEAN DATASET VALIDATION
Final Rows    : 456,087
Final Columns : 24
root
 |-- Age: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Race: string (nullable = false)
 |-- Marital_Status: string (nullable = false)
 |-- Tumor_Size: integer (nullable = true)
 |-- Vital_Status: string (nullable = true)
 |-- Grade: string (nullable = false)
 |-- PR_Status: string (nullable = true)
 |-- ER_Status: string (nullable = true)
 |-- AJCC_T: string (nullable = true)
 |-- AJCC_N: string (nullable = true)
 |-- Regional_Nodes_Examined: integer (nullable = true)
 |-- Regional_Nodes_Positive: integer (nullable = true)
 |-- Sequence_Number: string (nullable = true)
 |-- Histologic_Type: integer (nullable = true)
 |-- Laterality: string (nullable = true)
 |-- Diagnostic_Confirmation: string (nullable = false)
 |-- AJCC_M: string (nullable = true)
 |-- Surgery_Primary_Site: integer (nullable = true)
 |-- Surgery_Other_Regional: string (nullable = true)
 |-- Surgery_Radiation_Sequence: str

In [66]:
# 20. Compare Before vs After | So sánh sự thay đổi kích thước

print("=" * 60)
print("20. BEFORE VS AFTER PREPROCESSING")
print("=" * 60)

before_rows = original_row_count
before_cols = original_column_count
after_rows = df.count()
after_cols = len(df.columns)

print(f"Before Preprocessing (Rows) : {before_rows:,}")
print(f"After Preprocessing (Rows)  : {after_rows:,}")
print(f"Rows Removed                : {before_rows - after_rows:,}\n")

print(f"Before Preprocessing (Cols) : {before_cols}")
print(f"After Preprocessing (Cols)  : {after_cols}")
print(f"Columns Removed             : {before_cols - after_cols}")

20. BEFORE VS AFTER PREPROCESSING
Before Preprocessing (Rows) : 457,351
After Preprocessing (Rows)  : 456,087
Rows Removed                : 1,264

Before Preprocessing (Cols) : 29
After Preprocessing (Cols)  : 24
Columns Removed             : 5


In [67]:
# 21. Data Cleaning Report | Báo cáo tóm tắt quy trình làm sạch

print("=" * 60)
print("21. DATA CLEANING REPORT")
print("=" * 60)

print("""
Successfully completed data preprocessing pipeline:
✓ Standardized SEER column names
✓ Converted SEER special codes to NULL
✓ Filled missing categorical values with 'Unknown'
✓ Filtered invalid SEER records
✓ Performed statistical inspection
✓ Standardized numeric fields to Integer
✓ Consolidated raw labels to 'Unknown'
✓ Identified constant features for review
✓ Dropped 'Survival_Months' to prevent Target Leakage
✓ Dropped unused metadata columns
""")

21. DATA CLEANING REPORT

Successfully completed data preprocessing pipeline:
✓ Standardized SEER column names
✓ Converted SEER special codes to NULL
✓ Filled missing categorical values with 'Unknown'
✓ Filtered invalid SEER records
✓ Performed statistical inspection
✓ Standardized numeric fields to Integer
✓ Consolidated raw labels to 'Unknown'
✓ Identified constant features for review
✓ Dropped 'Survival_Months' to prevent Target Leakage
✓ Dropped unused metadata columns



In [68]:
# 22. Export Clean Dataset | Xuất tập dữ liệu sạch ra đĩa cứng

print("=" * 60)
print("22. EXPORT CLEAN DATASET")
print("=" * 60)

# Configure output directory structure | Cấu hình cấu trúc thư mục lưu đầu ra
output_dir = "../data/processed/seer_breast_cancer_clean"
os.makedirs(output_dir, exist_ok=True)

# Synchronize file name precisely for Notebook 03 | Đồng bộ chính xác tên file nạp của Notebook 03
output_file = os.path.join(output_dir, "seer_breast_cancer_clean.csv")

print("Converting DataFrame to Pandas and writing output CSV...")
# Convert to Pandas for local environment exporting without Winutils | Chuyển Pandas để ghi cục bộ không cần Winutils
df_pd = df.toPandas()
df_pd.to_csv(output_file, index=False, encoding="utf-8")

print(f"Cleaned dataset successfully exported to: {output_file}")

22. EXPORT CLEAN DATASET
Converting DataFrame to Pandas and writing output CSV...
Cleaned dataset successfully exported to: ../data/processed/seer_breast_cancer_clean\seer_breast_cancer_clean.csv


In [69]:
# 23. Preprocessing Summary | Tổng kết trạng thái vận hành

print("=" * 60)
print("23. PREPROCESSING SUMMARY")
print("=" * 60)

print("\nDataset: SEER Breast Cancer")
print("\nRaw Data")
print("-" * 25)
print(f"Rows    : {before_rows:,}")
print(f"Columns : {before_cols}")

print("\nClean Data")
print("-" * 25)
print(f"Rows    : {after_rows:,}")
print(f"Columns : {after_cols}")

print("\nDelta")
print("-" * 25)
print(f"Rows dropped    : {before_rows - after_rows:,}")
print(f"Columns dropped : {before_cols - after_cols}")

print("\nExecution Status")
print("-" * 25)
print("✓ Data preprocessing stage finished successfully.")
print("\nReady for Notebook 03 — Feature Engineering!")

23. PREPROCESSING SUMMARY

Dataset: SEER Breast Cancer

Raw Data
-------------------------
Rows    : 457,351
Columns : 29

Clean Data
-------------------------
Rows    : 456,087
Columns : 24

Delta
-------------------------
Rows dropped    : 1,264
Columns dropped : 5

Execution Status
-------------------------
✓ Data preprocessing stage finished successfully.

Ready for Notebook 03 — Feature Engineering!
